# idex gutendex api publique et api local

In [10]:
import runpy
import time
from pathlib import Path

runpy.run_path(str(next(p / "ipynb.py" for p in (Path.cwd(), *Path.cwd().parents) if (p / "ipynb.py").is_file())), init_globals=globals())

import matplotlib.pyplot as plt

from nlp_pipeline import config
from nlp_pipeline.retrieval import http_client

BOOK_ID = 11

In [ ]:
LOCAL_TIMEOUT = 5

def mesurer(url_template: str, book_id: int, *, timeout: int = config.HTTP_TIMEOUT) -> tuple[float | None, str]:
    """retourne (secondes, message)"""
    url = url_template.format(id=book_id)
    debut = time.perf_counter()
    try:
        status, data = http_client.get_json(url, headers=config.HTTP_HEADERS, timeout=timeout)
    except http_client.HttpError as exc:
        return None, f"indisponible ({exc})"
    duree = time.perf_counter() - debut
    if status != 200:
        return None, f"statut {status}"
    return duree, data.get("title", "?")

In [12]:
labels = ["API publique", "API locale"]
temps_ms: list[float | None] = []
details: list[str] = []

for label, url in zip(labels, config.GUTENDEX_URLS):
    timeout = LOCAL_TIMEOUT if label == "API locale" else config.HTTP_TIMEOUT
    duree, info = mesurer(url, BOOK_ID, timeout=timeout)
    temps_ms.append(duree * 1000 if duree is not None else None)
    details.append(info)
    if duree is None:
        print(f"{label} : {info}")
    else:
        print(f"{label} : {duree * 1000:.0f} ms ({info})")

API publique : indisponible (la requete a echoue (HTTPSConnectionPool(host='gutendex.com', port=443): Read timed out. (read timeout=30)))
API locale : indisponible (la requete a echoue (HTTPConnectionPool(host='192.168.1.22', port=8002): Max retries exceeded with url: /books/11 (Caused by ConnectTimeoutError(<HTTPConnection(host='192.168.1.22', port=8002) at 0x2d0f1ba0>, 'Connection to 192.168.1.22 timed out. (connect timeout=5)'))))


In [ ]:
ok = [(l, t) for l, t in zip(labels, temps_ms) if t is not None]
if not ok:
    print("off")
else:
    noms, valeurs = zip(*ok)
    plt.bar(noms, valeurs, color=["#4C78A8", "#F58518"][: len(noms)])
    plt.ylabel("temps (ms)")
    plt.title(f"gutendex livre #{BOOK_ID}")
    plt.tight_layout()
    plt.show()

off
